In [ ]:
from pathlib import Path

import pandas as pd 
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"
print(pio.renderers.default)
print(pio.renderers)

In [ ]:
train_dir = Path('../data/raw/train')
wells_list = list(train_dir.glob('*_horizontal_well.csv'))

well_sel = wells_list[0]

In [ ]:
def load_well_data(filepath):
    well_name = filepath.stem.split('__horizontal_well')[0]
    
    df_hor = pd.read_csv(filepath)
    df_vert = pd.read_csv(filepath.parent / f"{well_name}__typewell.csv")

    ref_img = plt.imread(filepath.parent / f"{well_name}.png", format=None)

    return well_name, df_hor, df_vert, ref_img

# Initial exploration

In [ ]:
well_name, df_hor, df_vert, ref_img = load_well_data(well_sel)

In [ ]:
plt.figure(figsize=(12,8))
plt.imshow(ref_img);

In [ ]:
df_hor

In [ ]:
df_hor.info()

In [ ]:
sns.heatmap(df_hor.isna())

In [ ]:
df_hor.describe()

In [ ]:
df_vert

In [ ]:
df_vert.info()

# Tests with merged data

In [ ]:
df_hor = pd.read_parquet('../data/merged_raw/train/horizontal_wells.parquet')
df_vert = pd.read_parquet('../data/merged_raw/train/vertical_wells.parquet')

## Missing values

In [ ]:
df_hor.info(show_counts=True)

In [ ]:
df_hor.isna().sum().sort_values(ascending=False)

In [ ]:
df_vert.isna().sum().sort_values(ascending=False)

## Understanding layers and TVT

In [ ]:
unique_wells = df_hor.well_id.unique()

for idx_well in range(10):
    selected_well = unique_wells[idx_well]
    
    df_sel = df_hor.loc[df_hor.well_id == selected_well]
    idx_target = df_sel['TVT_input'].isna()
    
    plt.scatter(df_sel["Z"], df_sel["TVT"], c=df_sel["MD"], label='all data')
    plt.plot(df_sel.loc[idx_target, "Z"], df_sel.loc[idx_target, "TVT"], c='r', label='target')
    plt.grid()
    plt.xlabel('Z')
    plt.ylabel('TVT')
    plt.colorbar()
    plt.legend()
    plt.show()

In [ ]:
df_sel.head()

In [ ]:
layer_cols = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

for idx_well in range(1):
    selected_well = unique_wells[idx_well]
    
    df_sel = df_hor.loc[df_hor.well_id == selected_well]
    df_sel_vert = df_vert.loc[df_vert.well_id == selected_well]
    idx_target = df_sel['TVT_input'].isna()

    df_sel_vert_layers = df_vert.drop_duplicates(subset=['Geology']).dropna().set_index('Geology')
    df_sel_vert_layers['TVT_diff_BUDA'] = -1*(df_sel_vert_layers['TVT'] - df_sel_vert_layers.loc['BUDA', 'TVT'])
    df_sel_vert_layers = df_sel_vert_layers.loc[df_sel_vert_layers.index.isin(layer_cols)]

    plt.figure(figsize=(20,8))
    plt.subplot(1,3,1)
    plt.scatter(df_sel["MD"], df_sel["Z"], label='all data')
    plt.plot(df_sel.loc[idx_target, "MD"], df_sel.loc[idx_target, "Z"], c='r', label='target')
    plt.plot(df_sel["MD"], df_sel["ANCC"], label='ANCC')
    plt.plot(df_sel["MD"], df_sel["ASTNU"], label='ASTNU')
    plt.plot(df_sel["MD"], df_sel["ASTNL"], label='ASTNL')
    plt.plot(df_sel["MD"], df_sel["EGFDU"], label='EGFDU')
    plt.plot(df_sel["MD"], df_sel["EGFDL"], label='EGFDL')
    plt.plot(df_sel["MD"], df_sel["BUDA"], label='BUDA')
    plt.grid()
    plt.xlabel('MD')
    plt.ylabel('Z')
    plt.colorbar()
    plt.legend()

    plt.subplot(1,3,2)
    plt.plot(df_sel["MD"], df_sel["Z"] - df_sel["BUDA"], label='Z - BUDA')
    plt.plot(df_sel["MD"], df_sel["ANCC"] - df_sel["BUDA"], label='ANCC - BUDA')
    plt.plot(df_sel["MD"], df_sel["ASTNU"] - df_sel["BUDA"], label='ASTNU - BUDA')
    plt.plot(df_sel["MD"], df_sel["ASTNL"] - df_sel["BUDA"], label='ASTNL - BUDA')
    plt.plot(df_sel["MD"], df_sel["EGFDU"] - df_sel["BUDA"], label='EGFDU - BUDA')
    plt.plot(df_sel["MD"], df_sel["EGFDL"] - df_sel["BUDA"], label='EGFDL - BUDA')
    plt.plot(df_sel["MD"], df_sel["BUDA"] - df_sel["BUDA"], label='BUDA - BUDA')
    plt.scatter(np.ones(len(df_sel_vert_layers['TVT_diff_BUDA'])) * df_sel["MD"].iloc[0], df_sel_vert_layers['TVT_diff_BUDA'], label='typewell layers to BUDA')
    plt.grid()
    plt.xlabel('MD')
    plt.ylabel('Z')
    plt.legend()
    
    plt.show()

## 3d plots for multiple wells 
- (attention - might have performance issues due to high amount of points)

In [ ]:
# Plot all wells in 3d grid
fig = px.scatter_3d(df_hor.dropna().iloc[::30], x='X', y='Y', z='Z', color='well_id')
fig.show()

In [ ]:
# plot layers in 3d grid

unique_wells = df_hor.well_id.unique()
selected_wells = unique_wells[:150]

layer_cols = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

df_sel = (
    df_hor.loc[df_hor.well_id.isin(selected_wells)]
    .iloc[::30]
    .dropna(subset=layer_cols + ['X', 'Y'])
)

# reshape
df_sel = df_sel.melt(
    id_vars=['X', 'Y', 'well_id'],
    value_vars=layer_cols,
    var_name='layer',
    value_name='Z_layer'
).dropna()

fig = px.scatter_3d(df_sel, 
                    x='X', y='Y', z='Z_layer', color='layer')
fig.show()